![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/Certification_Trainings/Healthcare/1.4.Resume_MedicalNer_Model_Training.ipynb)

If you are using the `johnsnowlabs` library, please use this  [01.2.Resume_MedicalNer_Model_Training](https://github.com/JohnSnowLabs/spark-nlp-workshop/blob/master/healthcare-nlp/01.2.Resume_MedicalNer_Model_Training.ipynb
) notebook.

# 1.4 Resume MedicalNer Model Training

Steps:
- Train a new model for a few epochs.
- Load the same model and train for more epochs on the same taxnonomy, and check stats.
- Train a model already trained on a different data

## Healthcare NLP for Data Scientists Course

If you are not familiar with the components in this notebook, you can check [Healthcare NLP for Data Scientists Udemy Course](https://www.udemy.com/course/healthcare-nlp-for-data-scientists/) and the [MOOC Notebooks](https://github.com/JohnSnowLabs/spark-nlp-workshop/tree/master/Spark_NLP_Udemy_MOOC/Healthcare_NLP) for each components.

## Colab Setup

In [ ]:
import json
import os

from google.colab import files

if 'spark_jsl.json' not in os.listdir():
  license_keys = files.upload()
  os.rename(list(license_keys.keys())[0], 'spark_jsl.json')

with open('spark_jsl.json') as f:
    license_keys = json.load(f)

# Defining license key-value pairs as local variables
locals().update(license_keys)
os.environ.update(license_keys)

In [ ]:
# Installing pyspark and spark-nlp
! pip install --upgrade -q pyspark==3.5.7  spark-nlp==$PUBLIC_VERSION

# Installing Spark NLP Healthcare
! pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION  --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Installing Spark NLP Display Library for visualization
! pip install -q spark-nlp-display

In [3]:
# if you want to start the session with custom params as in start function above
from pyspark.sql import SparkSession

def start(SECRET):
    builder = SparkSession.builder \
        .appName("Spark NLP Licensed") \
        .master("local[*]") \
        .config("spark.driver.memory", "16G") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.kryoserializer.buffer.max", "2000M") \
        .config("spark.jars.packages", "com.johnsnowlabs.nlp:spark-nlp_2.12:"+PUBLIC_VERSION) \
        .config("spark.jars", "https://pypi.johnsnowlabs.com/"+SECRET+"/spark-nlp-jsl-"+JSL_VERSION+".jar")

    return builder.getOrCreate()

#spark = start(SECRET)

In [4]:
import json
import os

from pyspark.ml import Pipeline,PipelineModel
from pyspark.sql import SparkSession

import sparknlp_jsl
import sparknlp

from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *


import warnings
warnings.filterwarnings('ignore')

params = {"spark.driver.memory":"16G", # Amount of memory to use for the driver process, i.e. where SparkContext is initialized
          "spark.kryoserializer.buffer.max":"2000M", # Maximum allowable size of Kryo serialization buffer, in MiB unless otherwise specified.
          "spark.driver.maxResultSize":"2000M"} # Limit of total size of serialized results of all partitions for each Spark action (e.g. collect) in bytes.
                                                # Should be at least 1M, or 0 for unlimited.

spark = sparknlp_jsl.start(license_keys['SECRET'],params=params)

print ("Spark NLP Version :", sparknlp.version())
print ("Spark NLP_JSL Version :", sparknlp_jsl.version())

spark

Spark NLP Version : 7.0.0
Spark NLP_JSL Version : 7.0.0


## Download Clinical Word Embeddings for training

In [5]:
clinical_embeddings = WordEmbeddingsModel.pretrained('embeddings_clinical', "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")

embeddings_clinical download started this may take some time.


Approximate size to download 1.6 GB
[ | ]

[ / ]

[ — ]

[ \ ]

[ | ]

... (25 outputs left out)

[ — ]

[ \ ]

[ | ]

[OK!]

## Download Data for Training (NCBI Disease Dataset)

In [6]:
!wget -q https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/NCBI_disease_official_test.conll
!wget -q https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/NCBI_disease_official_train_dev.conll

In [7]:
from sparknlp.training import CoNLL

training_data = CoNLL().readDataset(spark, 'NCBI_disease_official_train_dev.conll')

training_data.show(3)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text|            document|            sentence|               token|                 pos|               label|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Identification of...|[{document, 0, 89...|[{document, 0, 89...|[{token, 0, 13, I...|[{pos, 0, 13, NN,...|[{named_entity, 0...|
|The adenomatous p...|[{document, 0, 21...|[{document, 0, 21...|[{token, 0, 2, Th...|[{pos, 0, 2, NN, ...|[{named_entity, 0...|
|Complex formation...|[{document, 0, 63...|[{document, 0, 63...|[{token, 0, 6, Co...|[{pos, 0, 6, NN, ...|[{named_entity, 0...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [8]:
from sparknlp.training import CoNLL

test_data = CoNLL().readDataset(spark, 'NCBI_disease_official_test.conll')

test_data.show(3)

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|                text|            document|            sentence|               token|                 pos|               label|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|Clustering of mis...|[{document, 0, 10...|[{document, 0, 10...|[{token, 0, 9, Cl...|[{pos, 0, 9, NN, ...|[{named_entity, 0...|
|Ataxia - telangie...|[{document, 0, 13...|[{document, 0, 13...|[{token, 0, 5, At...|[{pos, 0, 5, NN, ...|[{named_entity, 0...|
|The risk of cance...|[{document, 0, 15...|[{document, 0, 15...|[{token, 0, 2, Th...|[{pos, 0, 2, NN, ...|[{named_entity, 0...|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



## Split the test data into two parts:
- We Keep the first part separate and use it for training the model further, as it will be totally unseen data from the same taxonomy.

- The second part will be used to testing and evaluating

In [9]:
(test_data_1, test_data_2) = test_data.randomSplit([0.5, 0.5], seed = 100)

# save the test data as parquet for easy testing
clinical_embeddings.transform(test_data_1).write.parquet('test_1.parquet')

clinical_embeddings.transform(test_data_2).write.parquet('test_2.parquet')

## Train a new model, pause, and resume training on the same dataset.

### MedicalNerDLGraphChecker for NER Training

The MedicalNerDLGraphChecker processes the dataset to extract required graph parameters (tokens, labels, embedding dimensions)

In [10]:
nerDLGraphChecker = MedicalNerDLGraphChecker()\
    .setInputCols(["sentence", "token"])\
    .setLabelColumn("label")\
    .setEmbeddingsModel(clinical_embeddings)

### Train for 2 epochs

In [11]:
nerTagger = MedicalNerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('./test_2.parquet')\
      .setOutputLogsPath('./ner_logs')

ner_pipeline = Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [12]:
%%time
ner_model = ner_pipeline.fit(training_data)

CPU times: user 208 ms, sys: 75.4 ms, total: 284 ms
Wall time: 8min 49s


In [13]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_f0c111f84619.log


In [14]:
# Training Logs
! cat ner_logs/MedicalNerApproach*

Name of the selected graph: medical-ner-dl/blstm_10_200_128_100.pb
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 6347


Epoch 1/2 started, lr: 0.003, dataset size: 6347


Epoch 1/2 - 253.96s - loss: 2062.9055 - avg training loss: 2.5948498 - batches: 795
Quality on test dataset: 
time to finish evaluation: 6.84s
Total test loss: 75.6613	Avg test loss: 1.2824
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 498	 134	 38	 0.78797466	 0.9291045	 0.8527397
B-Disease	 399	 100	 65	 0.7995992	 0.85991377	 0.82866037
tp: 897 fp: 234 fn: 103 labels: 2
Macro-average	 prec: 0.7937869, rec: 0.89450914, f1: 0.8411435
Micro-average	 prec: 0.79310346, rec: 0.897, f1: 0.84185827


Epoch 2/2 started, lr: 0.0029850747, dataset size: 6347


Epoch 2/2 - 239.95s - loss: 826.51447 - avg training loss: 1.0396408 - batches: 795
Quality on test dataset: 
time to finish evaluation: 3.33s
Total test loss: 51.7384	Avg test loss: 0.8769
label	 tp	 fp	 fn	 p

In [15]:
# Logs of 4 consecutive epochs to compare with 2+2 epochs on separate datasets from same taxonomy

#!cat ner_logs/MedicalNerApproach_4d3d69967c3f.log

### Evaluate

In [16]:
from sparknlp_jsl.eval import NerDLMetrics
import pyspark.sql.functions as F

pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|389.0|89.0|71.0|460.0|   0.8138|0.8457|0.8294|
+-------+-----+----+----+-----+---------+------+------+



+------------------+
|             macro|
+------------------+
|0.8294243070362474|
+------------------+

None


+------------------+
|             micro|
+------------------+
|0.8294243070362474|
+------------------+

None


`ner_utils`: This new module is used after NER training to calculate metric chunkbase and plot training logs.

`evaluate`: if verbose, returns overall performance, as well as performance per chunk type; otherwise, simply returns overall precision, recall, f1 scores

`loss_plot`: Plots the figure of loss vs epochs

`get_charts` : Plots the figures of metrics ( precision, recall, f1) vs epochs

```
from sparknlp_jsl.training_log_parser import ner_log_parser
parser = ner_log_parser()

pred_df = ner_model.stages[2].transform(test_data)
pred_df = pred_df.select(F.explode(F.arrays_zip(pred_df.ner.result,
                                                pred_df.label.result)).alias("cols"))\
                 .select(F.expr("cols['0']").alias("prediction"),
                         F.expr("cols['1']").alias("ground_truth"))
                 
df = pred_df.toPandas()

metrics = parser.evaluate( df['prediction'].values, df['ground_truth'].values)

parser.loss_plot(f"./ner_logs/{log_file}")

parser.get_charts('./ner_logs/'+log_file)
```

### Save the model to disk

In [17]:
ner_model.stages[2].write().overwrite().save('models/NCBI_NER_2_epoch')

### Train using the saved model on unseen dataset
#### We use unseen data from same taxonomy

In [18]:
nerTagger = MedicalNerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('/content/test_2.parquet')\
      .setOutputLogsPath('ner_logs')\
      .setPretrainedModelPath("models/NCBI_NER_2_epoch") ## load existing model

ner_pipeline = Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [19]:
# remove the existing logs

! rm -r ./ner_logs

In [20]:
%%time
ner_model_retrained = ner_pipeline.fit(test_data_1)

CPU times: user 75.2 ms, sys: 17.6 ms, total: 92.8 ms
Wall time: 1min


In [21]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_321bf3cabe4f.log


In [22]:
!cat ./ner_logs/MedicalNerApproach*

Name of the selected graph: pretrained graph
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 477


Epoch 1/2 started, lr: 0.003, dataset size: 477


Epoch 1/2 - 25.40s - loss: 74.4413 - avg training loss: 1.2203492 - batches: 61
Quality on test dataset: 
time to finish evaluation: 3.82s
Total test loss: 43.8025	Avg test loss: 0.7424
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 477	 67	 59	 0.8768382	 0.88992536	 0.8833333
B-Disease	 417	 61	 47	 0.87238497	 0.8987069	 0.8853503
tp: 894 fp: 128 fn: 106 labels: 2
Macro-average	 prec: 0.8746116, rec: 0.89431614, f1: 0.8843541
Micro-average	 prec: 0.8747554, rec: 0.894, f1: 0.88427293


Epoch 2/2 started, lr: 0.0029850747, dataset size: 477


Epoch 2/2 - 20.68s - loss: 46.2491 - avg training loss: 0.758182 - batches: 61
Quality on test dataset: 
time to finish evaluation: 3.12s
Total test loss: 40.4055	Avg test loss: 0.6848
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 505	 118	 31	 

In [23]:
from sparknlp_jsl.eval import NerDLMetrics
import pyspark.sql.functions as F

pred_df = ner_model_retrained.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|407.0|77.0|53.0|460.0|   0.8409|0.8848|0.8623|
+-------+-----+----+----+-----+---------+------+------+



+------------------+
|             macro|
+------------------+
|0.8622881355932204|
+------------------+

None


+------------------+
|             micro|
+------------------+
|0.8622881355932204|
+------------------+

None


### Train a Model Choosing Best Model

`useBestModel`:  This param preserves and restores the model that has achieved the best performance at the end of the training. The priority is metrics from testDataset (micro F1), metrics from validationSplit (micro F1), and if none is set it will keep track of loss during the training. <br/>

`earlyStopping`: You can stop training at the point when the perforfmance on test/validation dataset starts to degrage. Two params are used in order to use this feature: <br/>

- `earlyStoppingCriterion`: This is used set the minimal improvement of the test metric to terminate training. The metric monitored is the same as the metrics used in `useBestModel` (micro F1 when using test/validation set, loss otherwise). Default is 0 which means no early stopping is applied.

- `earlyStoppingPatience`:  The number of epoch without improvement which will be tolerated. Default is 0, which means that early stopping will occur at the first time when performance in the current epoch is no better than in the previous epoch.

In [24]:
# remove the existing logs

! rm -r ./ner_logs

In [25]:
nerTagger = MedicalNerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(20)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('./test_2.parquet')\
      .setOutputLogsPath('./ner_logs')\
      .setValidationSplit(0.2)\
      .setUseBestModel(True)\
      .setEarlyStoppingCriterion(0.04)\
      .setEarlyStoppingPatience(3)


ner_pipeline = Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [26]:

%%time
ner_model = ner_pipeline.fit(training_data)

CPU times: user 279 ms, sys: 97.9 ms, total: 377 ms
Wall time: 14min 32s


In [27]:
!ls ner_logs/MedicalNerApproach*

ner_logs/MedicalNerApproach_040fe706b396.log


In [28]:
# Training Logs
! cat ner_logs/MedicalNerApproach*

Name of the selected graph: medical-ner-dl/blstm_10_200_128_100.pb
Training started - total epochs: 20 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 84 - training examples: 5078


Epoch 1/20 started, lr: 0.003, dataset size: 5078


Epoch 1/20 - 210.13s - loss: 1653.1658 - avg training loss: 2.5871139 - batches: 639
Quality on validation dataset (20.0%), validation examples = 1269
time to finish evaluation: 11.55s
Total validation loss: 179.9310	Avg validation loss: 1.1316
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 1289	 121	 256	 0.9141844	 0.8343042	 0.8724196
B-Disease	 972	 136	 223	 0.87725633	 0.8133891	 0.8441164
tp: 2261 fp: 257 fn: 479 labels: 2
Macro-average	 prec: 0.89572036, rec: 0.8238467, f1: 0.85828143
Micro-average	 prec: 0.89793485, rec: 0.8251825, f1: 0.86002284
Quality on test dataset: 
time to finish evaluation: 3.17s
Total test loss: 63.8905	Avg test loss: 1.0829
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 458	 61	 78	 0.88246626	 0.8544776	 0.86824644
B-Disease

As you see above, training was stopped before 20th epoch since there were not improvement.

**Evaluate**

In [29]:
from sparknlp_jsl.eval import NerDLMetrics
import pyspark.sql.functions as F

pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_1))

evaler = NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|426.0|73.0|69.0|495.0|   0.8537|0.8606|0.8571|
+-------+-----+----+----+-----+---------+------+------+



+------------------+
|             macro|
+------------------+
|0.8571428571428571|
+------------------+

None


+------------------+
|             micro|
+------------------+
|0.8571428571428571|
+------------------+

None


In [30]:
pred_df = ner_model.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|393.0|67.0|67.0|460.0|   0.8543|0.8543|0.8543|
+-------+-----+----+----+-----+---------+------+------+



+------------------+
|             macro|
+------------------+
|0.8543478260869565|
+------------------+

None


+------------------+
|             micro|
+------------------+
|0.8543478260869565|
+------------------+

None


## Now let's take a model trained on a different dataset and train on this dataset

In [31]:
jsl_ner = MedicalNerModel.pretrained('ner_jsl','en','clinical/models')

jsl_ner.getClasses()

ner_jsl download started this may take some time.


Approximate size to download 14.5 MB
[ | ]

[ / ]

[OK!]

['O',
 'B-Injury_or_Poisoning',
 'B-Direction',
 'B-Test',
 'I-Route',
 'B-Admission_Discharge',
 'B-Death_Entity',
 'I-Oxygen_Therapy',
 'B-Relationship_Status',
 'I-Drug_BrandName',
 'B-Duration',
 'I-Alcohol',
 'I-Triglycerides',
 'I-Date',
 'B-Hyperlipidemia',
 'B-Respiration',
 'I-Test',
 'B-Birth_Entity',
 'I-VS_Finding',
 'B-Age',
 'I-Vaccine_Name',
 'I-Social_History_Header',
 'B-Labour_Delivery',
 'I-Medical_Device',
 'B-Family_History_Header',
 'B-BMI',
 'I-Fetus_NewBorn',
 'I-BMI',
 'B-Temperature',
 'I-Section_Header',
 'I-Communicable_Disease',
 'I-ImagingFindings',
 'I-Psychological_Condition',
 'I-Obesity',
 'I-Sexually_Active_or_Sexual_Orientation',
 'I-Modifier',
 'B-Alcohol',
 'I-Temperature',
 'I-Vaccine',
 'I-Symptom',
 'I-Pulse',
 'B-Kidney_Disease',
 'B-Oncological',
 'I-EKG_Findings',
 'B-Medical_History_Header',
 'I-Relationship_Status',
 'B-Cerebrovascular_Disease',
 'I-Blood_Pressure',
 'I-Diabetes',
 'B-Oxygen_Therapy',
 'B-O2_Saturation',
 'B-Psychological_C

### Now train a model using this model as base

In [32]:
nerTagger = MedicalNerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setTestDataset('/content/test_2.parquet')\
      .setOutputLogsPath('ner_logs')\
      .setPretrainedModelPath("/root/cache_pretrained/ner_jsl_en_4.2.0_3.0_1666181370373")\
      .setOverrideExistingTags(True) # since the tags do not align, set this flag to true

# do hyperparameter by tuning the params above (max epoch, LR, dropout etc.) to get better results
ner_pipeline = Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger
 ])

In [33]:
# remove the existing logs

! rm -r ./ner_logs

In [34]:

%%time
ner_jsl_retrained = ner_pipeline.fit(training_data)


CPU times: user 233 ms, sys: 94.2 ms, total: 327 ms
Wall time: 11min 19s


In [35]:
!cat ./ner_logs/MedicalNerApproach*

Name of the selected graph: pretrained graph
Training started - total epochs: 2 - lr: 0.003 - batch size: 8 - labels: 3 - chars: 103 - training examples: 6347


Epoch 1/2 started, lr: 0.003, dataset size: 6347


Epoch 1/2 - 320.65s - loss: 1380.7274 - avg training loss: 1.7367641 - batches: 795
Quality on test dataset: 
time to finish evaluation: 21.15s
Total test loss: 72.3105	Avg test loss: 1.2256
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Disease	 507	 165	 29	 0.75446427	 0.9458955	 0.83940387
B-Disease	 408	 106	 56	 0.7937743	 0.87931037	 0.8343558
tp: 915 fp: 271 fn: 85 labels: 2
Macro-average	 prec: 0.77411926, rec: 0.9126029, f1: 0.83767617
Micro-average	 prec: 0.7715008, rec: 0.915, f1: 0.8371455


Epoch 2/2 started, lr: 0.0029850747, dataset size: 6347


Epoch 2/2 - 311.34s - loss: 675.4112 - avg training loss: 0.84957385 - batches: 795
Quality on test dataset: 
time to finish evaluation: 15.50s
Total test loss: 51.6775	Avg test loss: 0.8759
label	 tp	 fp	 fn	 prec	 rec	 f1
I-Diseas

In [36]:
pred_df = ner_jsl_retrained.stages[2].transform(clinical_embeddings.transform(test_data_2))

evaler = NerDLMetrics(mode="full_chunk")

eval_result = evaler.computeMetricsFromDF(pred_df.select("label","ner"), prediction_col="ner", label_col="label", drop_o = True, case_sensitive = True).cache()

eval_result.withColumn("precision", F.round(eval_result["precision"],4))\
           .withColumn("recall", F.round(eval_result["recall"],4))\
           .withColumn("f1", F.round(eval_result["f1"],4)).show(100)

print(eval_result.selectExpr("avg(f1) as macro").show())
print (eval_result.selectExpr("sum(f1*total) as sumprod","sum(total) as sumtotal").selectExpr("sumprod/sumtotal as micro").show())

+-------+-----+----+----+-----+---------+------+------+
| entity|   tp|  fp|  fn|total|precision|recall|    f1|
+-------+-----+----+----+-----+---------+------+------+
|Disease|397.0|90.0|63.0|460.0|   0.8152| 0.863|0.8384|
+-------+-----+----+----+-----+---------+------+------+



+------------------+
|             macro|
+------------------+
|0.8384371700105596|
+------------------+

None


+------------------+
|             micro|
+------------------+
|0.8384371700105596|
+------------------+

None


## Lower-Memory NER Training with `explodeSentencesForTraining` (6.4.0)

Healthcare NLP **6.4.0** adds `explodeSentencesForTraining` to `MedicalNerApproach`. When enabled, multi-sentence training rows are split into one row per sentence before training. This can reduce peak memory usage and improve training efficiency on long documents.

Validation splitting stays at the **document level** first; sentence rows are generated only for the training step. Default is `False`.

| explodeSentencesForTraining | Approx. training time | Approx. peak RAM |
|------------------------------|----------------------:|-----------------:|
| `True` | ~11 minutes | ~9.3 GB |
| `False` | ~24 minutes | ~30 GB |

Benchmark on 30 documents (~30,000 characters). Actual results vary with hardware, Spark configuration, and hyperparameters.


In [37]:
nerTagger_explode = MedicalNerApproach()\
      .setInputCols(["sentence", "token", "embeddings"])\
      .setLabelColumn("label")\
      .setOutputCol("ner")\
      .setMaxEpochs(2)\
      .setLr(0.003)\
      .setBatchSize(8)\
      .setRandomSeed(0)\
      .setVerbose(1)\
      .setValidationSplit(0.2)\
      .setEvaluationLogExtended(True) \
      .setEnableOutputLogs(True)\
      .setIncludeConfidence(True)\
      .setOutputLogsPath("./ner_logs")\
      .setExplodeSentencesForTraining(True)

explode_pipeline = Pipeline(stages=[
      clinical_embeddings,
      nerDLGraphChecker,
      nerTagger_explode,
])


In [38]:
%%time
ner_explode_model = explode_pipeline.fit(training_data)


CPU times: user 166 ms, sys: 63.6 ms, total: 229 ms
Wall time: 7min 34s
